In [0]:
# Databricks notebook source
# ══════════════════════════════════════
# GOLD — KPI MoM Receita por Tipo de Pagamento
# Squad 3 — Arquitetura Medalhao
# Substitui KPI 8 itens (pereciveis vs secos)
# Dados 100% internos — sem dependencias externas
# Frequencia: toda segunda-feira as 5h
# ══════════════════════════════════════

In [0]:
%run "../config/00_config"

In [0]:
%run "../utils/00_utils"

In [0]:
# constantes do notebook

SILVER_ITENS_PATH = f"{SILVER_BASE_PATH}physical_itens_venda_caixa"
SILVER_LOJAS_PATH = f"{SILVER_BASE_PATH}physical_lojas"

GOLD_KPI8ALT_PATH  = f"{GOLD_BASE_PATH}kpi_mom_receita_tipo_pagamento"
GOLD_KPI8ALT_TABLE = f"{TARGET_SCHEMA}.gold_kpi_mom_receita_tipo_pagamento"

GOLD_WRITE_MODE = "overwrite"

print("Constantes configuradas:")
print(f"   SILVER_ITENS_PATH  : {SILVER_ITENS_PATH}")
print(f"   SILVER_LOJAS_PATH  : {SILVER_LOJAS_PATH}")
print(f"   GOLD_KPI8ALT_PATH  : {GOLD_KPI8ALT_PATH}")
print(f"   GOLD_KPI8ALT_TABLE : {GOLD_KPI8ALT_TABLE}")

In [0]:
# configuracoes do ADLS

adls_options = get_adls_options()
print("Opcoes ADLS configuradas.")

In [0]:
# ler Silver itens e lojas

df_itens = read_delta(
    spark        = spark,
    path         = SILVER_ITENS_PATH,
    adls_options = adls_options
)
print(f"Silver itens : {df_itens.count():,} linhas")

df_lojas = read_delta(
    spark        = spark,
    path         = SILVER_LOJAS_PATH,
    adls_options = adls_options
)
print(f"Silver lojas : {df_lojas.count():,} linhas")

display(df_itens.limit(5))

In [0]:
# converter tipos necessarios para calculos

from pyspark.sql.functions import (
    col, count, sum as spark_sum,
    round as spark_round, when,
    lag, current_timestamp, trim, upper
)
from pyspark.sql.window import Window
from pyspark.sql.types import IntegerType, DoubleType

df_itens = (
    df_itens
    .withColumn("valor_total_item",
        col("valor_total_item").cast(DoubleType()))
    .withColumn("id_loja",
        col("id_loja").cast(IntegerType()))
    .withColumn("ano",
        col("ano").cast(IntegerType()))
    .withColumn("mes",
        col("mes").cast(IntegerType()))
    .withColumn("tipo_pagamento",
        trim(upper(col("tipo_pagamento"))))
)

df_lojas = (
    df_lojas
    .withColumn("id_loja",
        col("id_loja").cast(IntegerType()))
)

print("Tipos convertidos com sucesso!")

In [0]:
# verificar tipos de pagamento disponiveis

print("Tipos de pagamento disponiveis:")
display(
    df_itens
    .groupBy("tipo_pagamento")
    .count()
    .orderBy("tipo_pagamento")
)

In [0]:
# calcular receita por loja, tipo pagamento, ano e mes

df_receita_pagto = (
    df_itens
    .groupBy("id_loja", "tipo_pagamento", "ano", "mes")
    .agg(
        spark_round(
            spark_sum("valor_total_item"), 2
        ).alias("receita_total"),
        count("id_item_venda").alias("total_itens"),
        count(col("id_transacao")).alias("total_transacoes"),
    )
)

print(f"Receita por pagamento/loja/mes: {df_receita_pagto.count():,} linhas")
display(df_receita_pagto.orderBy("id_loja", "tipo_pagamento", "ano", "mes").limit(10))

In [0]:
# calcular crescimento MoM por loja e tipo de pagamento

window_mom = (
    Window
    .partitionBy("id_loja", "tipo_pagamento")
    .orderBy("ano", "mes")
)

df_kpi8alt = (
    df_receita_pagto
    .withColumn("receita_mes_anterior",
        lag("receita_total", 1).over(window_mom))
    .withColumn("total_itens_mes_anterior",
        lag("total_itens", 1).over(window_mom))
    .withColumn(
        "crescimento_mom_receita_pct",
        when(
            col("receita_mes_anterior").isNotNull() &
            (col("receita_mes_anterior") > 0),
            spark_round(
                (col("receita_total") - col("receita_mes_anterior")) /
                col("receita_mes_anterior") * 100,
                2
            )
        ).otherwise(None)
    )
    .withColumn(
        "crescimento_mom_itens_pct",
        when(
            col("total_itens_mes_anterior").isNotNull() &
            (col("total_itens_mes_anterior") > 0),
            spark_round(
                (col("total_itens") - col("total_itens_mes_anterior")) /
                col("total_itens_mes_anterior") * 100,
                2
            )
        ).otherwise(None)
    )
    .withColumn(
        "tendencia",
        when(col("crescimento_mom_receita_pct") > 0, "CRESCIMENTO")
        .when(col("crescimento_mom_receita_pct") < 0, "QUEDA")
        .when(col("crescimento_mom_receita_pct") == 0, "ESTAVEL")
        .otherwise("PRIMEIRO_MES")
    )
    .join(
        df_lojas.select(
            "id_loja", "nome_loja",
            "cidade_loja", "estado_loja"
        ),
        on="id_loja",
        how="left"
    )
    .withColumn("gold_processed_at", current_timestamp())
)

total_kpi8alt = df_kpi8alt.count()
print(f"KPI 8ALT — MoM tipo pagamento: {total_kpi8alt:,} linhas")
display(
    df_kpi8alt
    .orderBy("id_loja", "tipo_pagamento", "ano", "mes")
    .limit(10)
)

In [0]:
# analise de qualidade do KPI

print("=" * 55)
print("ANALISE DE QUALIDADE — KPI 8ALT")
print("=" * 55)

# distribuicao por tendencia
print("\nDistribuicao por tendencia:")
display(
    df_kpi8alt
    .groupBy("tendencia")
    .count()
    .orderBy("tendencia")
)

# tipos de pagamento por loja
print("\nTipos de pagamento por loja:")
display(
    df_kpi8alt
    .groupBy("id_loja", "nome_loja")
    .agg(
        count("tipo_pagamento").alias("combinacoes_mes"),
    )
    .orderBy("id_loja")
)

# verificar nulos em colunas criticas
nulos_receita = df_kpi8alt.filter(
    col("receita_total").isNull()
).count()

if nulos_receita > 0:
    print(f"Atencao: {nulos_receita} registros com receita_total nula.")
else:
    print("Validacao OK: nenhum valor nulo em receita_total.")

In [0]:
# top 5 tipos de pagamento com maior crescimento MoM

print("Top 5 — Maior crescimento MoM de receita:")
display(
    df_kpi8alt
    .filter(col("crescimento_mom_receita_pct").isNotNull())
    .orderBy(col("crescimento_mom_receita_pct").desc())
    .select(
        "nome_loja",
        "tipo_pagamento",
        "ano",
        "mes",
        "receita_total",
        "receita_mes_anterior",
        "crescimento_mom_receita_pct",
        "tendencia"
    )
    .limit(5)
)

print("\nTop 5 — Maior queda MoM de receita:")
display(
    df_kpi8alt
    .filter(col("crescimento_mom_receita_pct").isNotNull())
    .orderBy(col("crescimento_mom_receita_pct").asc())
    .select(
        "nome_loja",
        "tipo_pagamento",
        "ano",
        "mes",
        "receita_total",
        "receita_mes_anterior",
        "crescimento_mom_receita_pct",
        "tendencia"
    )
    .limit(5)
)

In [0]:
# gravar Gold Delta no ADLS

write_delta(
    df           = df_kpi8alt,
    path         = GOLD_KPI8ALT_PATH,
    mode         = GOLD_WRITE_MODE,
    partition_by = ["ano", "mes"],
    adls_options = adls_options
)

In [0]:
# gravar no SQL Server

write_sql_table(
    df         = df_kpi8alt,
    table_name = GOLD_KPI8ALT_TABLE,
    mode       = GOLD_WRITE_MODE
)

In [0]:
# COMMAND ----------

# validar carga Delta

df_kpi8alt_saved = read_delta(
    spark        = spark,
    path         = GOLD_KPI8ALT_PATH,
    adls_options = adls_options
)

compare_row_counts(
    source_df = df_kpi8alt,
    target_df = df_kpi8alt_saved,
    label     = "KPI 8ALT memoria x Delta gravado"
)